# Your First Agent with AutoGen

This notebook walks through building your **first AI agent** using Microsoft's [AutoGen](https://microsoft.github.io/autogen/) framework (v0.4+).

## What is AutoGen?
AutoGen is a framework for building LLM-powered applications using **agents** that can converse with each other and with humans. An *agent* in AutoGen is an entity that can:
- Send and receive messages
- Use tools (functions)
- Reason with an LLM
- Collaborate with other agents

## What we'll cover
1. Install and verify AutoGen
2. Configure an LLM client (OpenAI)
3. Build a single `AssistantAgent`
4. Send it a message and read the response
5. Give the agent a **tool** (a Python function it can call)
6. Run a short multi-turn task

> AutoGen v0.4 split the package into `autogen-core`, `autogen-agentchat`, and `autogen-ext`. We'll use the high-level `agentchat` API.

## 1. Install dependencies
Run this once. It installs everything from `requirements.txt` in this folder.

In [ ]:
# %pip install -r requirements.txt

## 2. Set up your API key

Create a `.env` file in this folder containing:
```
OPENAI_API_KEY=sk-...
```
We'll load it with `python-dotenv` so the key never appears in the notebook.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Add it to .env"
print("API key loaded.")

API key loaded.


## 3. Create the model client

An agent needs an LLM to think with. In AutoGen v0.4 the LLM is wrapped in a **model client**. Here we use OpenAI's `gpt-4o-mini` — cheap and fast, ideal for learning.

In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    # api_key is read automatically from OPENAI_API_KEY
)

print("Model client ready:", model_client.model_info)

Model client ready: {'vision': True, 'function_calling': True, 'json_output': True, 'family': 'gpt-4o', 'structured_output': True, 'multiple_system_messages': True}


## 4. Build your first agent

An `AssistantAgent` is the simplest useful agent. The four parameters that matter most:

| Parameter | Who reads it | Purpose |
|---|---|---|
| `name` | Logs, message routing | Unique identifier (no spaces) |
| `model_client` | Internal | The LLM that powers the agent |
| `description` | **Other agents** / team coordinators | Short, third-person summary of *what this agent is good at*. Used by `SelectorGroupChat` and humans to decide when to route a task to it. The agent does **not** read its own description. |
| `system_message` | **The agent itself** (every turn) | Persona, rules, tone, constraints |

> Rule of thumb: `description` is the agent's **résumé** (seen by others), `system_message` is its **inner monologue** (seen by itself).
>
> If you only have one agent, `description` is optional — but get in the habit of writing it now, because the moment you build a team it becomes critical.

In [3]:
from autogen_agentchat.agents import AssistantAgent

assistant = AssistantAgent(
    name="helper",
    model_client=model_client,
    description="A friendly tutor that explains AI and programming concepts in simple language with short examples.",
    system_message=(
        "You are a friendly tutor. Explain concepts simply, "
        "using short examples. Keep replies under 5 sentences."
    ),
)

print(f"Name:        {assistant.name}")
print(f"Description: {assistant.description}")

Name:        helper
Description: A friendly tutor that explains AI and programming concepts in simple language with short examples.


## 5. Talk to your agent

AutoGen v0.4 is async-first. We use `await agent.run(task=...)` from inside the notebook (Jupyter already has an event loop).

The result is a `TaskResult` containing every message produced during the run.

In [4]:
from autogen_agentchat.messages import TextMessage

result = await assistant.run(task="In one paragraph, what is an AI agent?")

for msg in result.messages:
    print(f"[{msg.source}] {msg.content}\n")

[user] In one paragraph, what is an AI agent?

[helper] An AI agent is a computer system that can perceive its environment, make decisions, and take actions to achieve specific goals. For example, a chatbot is an AI agent that understands user input and responds appropriately to help with questions or tasks. It observes the text it receives, processes it using algorithms, and decides on the best reply. Essentially, AI agents act autonomously or semi-autonomously to perform functions that typically require human intelligence.



## 6. Give the agent a tool

A **tool** is just a Python function the agent can decide to call. The model sees the function's name, signature, and docstring, and calls it with arguments when useful.

Below: a calculator tool. Notice we keep it focused, with **type hints** and a **clear docstring** — these are how the LLM understands what the tool does.

In [ ]:
def add(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the product."""
    return a * b

math_agent = AssistantAgent(
    name="mathbot",
    model_client=model_client,
    description="A math assistant that performs accurate arithmetic by calling calculator tools (add, multiply).",
    tools=[add, multiply],
    system_message=(
        "You are a math assistant. ALWAYS call the provided tools "
        "to compute results — never do arithmetic in your head."
    ),
    reflect_on_tool_use=True,
)

print(f"Name:        {math_agent.name}")
print(f"Description: {math_agent.description}\n")

await Console(math_agent.run_stream(task="What is (12 + 8) multiplied by 4?"))

Watch the trace above — you should see:
1. The agent decides to call `add(12, 8)` → `20`
2. Then it calls `multiply(20, 4)` → `80`
3. With `reflect_on_tool_use=True`, the agent writes a final natural-language answer summarizing the result.

## 7. Multi-turn conversation

Agents keep **memory of the conversation** across calls to `run`. Each new `task` is appended to the existing history.

In [ ]:
await Console(assistant.run_stream(task="My name is Pradeep and I'm learning AutoGen."))

In [ ]:
await Console(assistant.run_stream(task="What's my name and what am I learning?"))

If you want a clean slate, reset the agent:

In [ ]:
await assistant.on_reset(cancellation_token=None)
print("Agent memory cleared.")

## 8. Clean up
Always close the model client when done — it holds an HTTP session.

In [ ]:
await model_client.close()
print("Done.")

## Recap & what's next

You just built:
- A single-LLM **AssistantAgent** with a system prompt
- A **tool-using agent** that calls Python functions
- A **streaming**, **multi-turn** conversation

Try changing the `system_message`, swap the model to `gpt-4o`, or add a new tool — and see how the agent's behavior changes.